# 🧠 พื้นที่ใต้เส้นโค้ง ROC (Area Under the ROC Curve - AUC)

ยินดีต้อนรับสู่สมุดโน้ตอธิบายการใช้งานจริงสำหรับ **AUC**! ในสมุดโน้ตเล่มนี้ เราจะ:
1. กำหนดความหมายทางคณิตศาสตร์และความหมายเชิงความน่าจะเป็นของ AUC
2. อิมพลีเมนต์ **วิธีการคำนวณสองรูปแบบ** เพื่อหาค่า AUC จากศูนย์ (from scratch):
   - **วิธีที่ 1 (Trapezoidal Rule Integration):** การอินทิเกรตพิกัดของเส้นโค้ง ROC โดยใช้พื้นที่รูปสี่เหลี่ยมคางหมู
   - **วิธีที่ 2 (Probabilistic Verification):** การเปรียบเทียบข้อมูลแบบจับคู่ระหว่าง Positive Class และ Negative Class เพื่อคำนวณ:
     $$P(f(x^+) > f(x^-))$$
3. สร้างการกระจายความน่าจะเป็นของแต่ละคลาส และแสดงให้เห็นภาพว่าการคาบเกี่ยวกันของคลาสส่งผลโดยตรงต่อค่า AUC อย่างไร
4. ตรวจสอบความถูกต้องของการอิมพลีเมนต์ของเราเทียบกับ `scikit-learn`
5. เชื่อมโยงแนวคิด AUC เข้ากับ **mean Average Precision (mAP)** ในงานตรวจจับวัตถุ (Object Detection)

เรามาเริ่มด้วยการนำเข้าไลบรารีที่จำเป็นกันเลยครับ

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score

# Set seed for reproducibility
np.random.seed(42)

## 1. การกระจายตัวของคลาสและค่า AUC (Class Distributions and AUC)
เราจะสร้างชุดผลการทำนายสามชุดที่แสดงถึงระดับความสามารถในการแยกคลาสของโมเดลในระดับต่างๆ ดังนี้ครับ:
1.  **Perfect Model:** การแยกคลาสได้สมบูรณ์แบบ 100%
2.  **Realistic Model:** มีความคาบเกี่ยวกันปานกลาง (รูปแบบที่พบได้จริงทั่วไป)
3.  **Random Model:** คลาสทับซ้อนกันโดยสิ้นเชิง (สุ่มเดา)

In [ ]:
n_samples = 100
y_true = np.concatenate([np.ones(n_samples), np.zeros(n_samples)]).astype(int)

# Perfect Model
scores_perfect = np.concatenate([
    np.random.normal(0.85, 0.05, n_samples),
    np.random.normal(0.15, 0.05, n_samples)
])

# Realistic Model
scores_real = np.concatenate([
    np.random.normal(0.65, 0.15, n_samples),
    np.random.normal(0.35, 0.15, n_samples)
])

# Random Model
scores_random = np.concatenate([
    np.random.normal(0.5, 0.15, n_samples),
    np.random.normal(0.5, 0.15, n_samples)
])

# Clip scores to [0.0, 1.0]
scores_perfect = np.clip(scores_perfect, 0, 1)
scores_real = np.clip(scores_real, 0, 1)
scores_random = np.clip(scores_random, 0, 1)

เรามาพล็อตฮิสโตแกรมของคะแนน (score histograms) ของโมเดลเหล่านี้กันครับ

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

models = {
    'Perfect Model': scores_perfect,
    'Realistic Model': scores_real,
    'Random Model': scores_random
}

for idx, (name, scores) in enumerate(models.items()):
    ax = axes[idx]
    ax.hist(scores[y_true == 1], color='blue', alpha=0.5, bins=15, label='Class 1: Positives')
    ax.hist(scores[y_true == 0], color='red', alpha=0.5, bins=15, label='Class 0: Negatives')
    ax.set_title(f"{name}\nROC AUC: {roc_auc_score(y_true, scores):.3f}")
    ax.set_xlabel('Predicted Probability')
    ax.set_ylabel('Frequency')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. การอิมพลีเมนต์การคำนวณ AUC จากศูนย์ (from Scratch)

### วิธีที่ 1: การอินทิเกรตโดยใช้กฎสี่เหลี่ยมคางหมู (Integration using the Trapezoidal Rule)
เมื่อเราได้จุดพิกัด ROC `(fpr, tpr)` แล้ว เราจะอินทิเกรตโดยการคำนวณพื้นที่ของรูปสี่เหลี่ยมคางหมูที่อยู่ติดกัน:
$$\text{Area} = \sum_{i=1}^{k} \frac{1}{2} (\text{tpr}_i + \text{tpr}_{i-1}) \cdot (\text{fpr}_i - \text{fpr}_{i-1})$$

### วิธีที่ 2: การเปรียบเทียบความน่าจะเป็นแบบจับคู่ (Pairwise Probability Comparison)
เราจะเปรียบเทียบตัวอย่างบวกทั้งหมด $x^+$ กับตัวอย่างลบทั้งหมด $x^-$ หาก $f(x^+) > f(x^-)$ เราจะบวกค่า 1 หากมีค่าเท่ากัน เราจะบวกค่า 0.5 ผลรวมทั้งหมดหารด้วย $|P| \times |N|$ จะเป็นค่า AUC ครับ

In [ ]:
def custom_roc_curve(y_true, scores):
    thresholds = np.sort(scores)[::-1]
    thresholds = np.concatenate([[1.001], thresholds])
    tprs, fprs = [], []
    for thresh in thresholds:
        y_pred = (scores >= thresh).astype(int)
        TP = np.sum((y_true == 1) & (y_pred == 1))
        TN = np.sum((y_true == 0) & (y_pred == 0))
        FP = np.sum((y_true == 0) & (y_pred == 1))
        FN = np.sum((y_true == 1) & (y_pred == 0))
        tprs.append(TP / (TP + FN) if (TP + FN) > 0 else 0.0)
        fprs.append(FP / (TN + FP) if (TN + FP) > 0 else 0.0)
    return np.array(fprs), np.array(tprs)

# Method 1: Trapezoidal Integration
def custom_auc_integration(fpr, tpr):
    """
    Calculate Area Under the Curve using the trapezoidal rule.
    """
    area = 0.0
    for i in range(1, len(fpr)):
        height = tpr[i] + tpr[i-1]
        width = fpr[i] - fpr[i-1]
        area += 0.5 * height * width
    return area

# Method 2: Pairwise Comparisons
def custom_auc_probabilistic(y_true, scores):
    """
    Calculate AUC as the probability of ranking a positive sample higher than a negative sample.
    """
    pos_scores = scores[y_true == 1]
    neg_scores = scores[y_true == 0]
    
    comparisons = 0.0
    for p in pos_scores:
        for n in neg_scores:
            if p > n:
                comparisons += 1.0
            elif p == n:
                comparisons += 0.5
                
    return comparisons / (len(pos_scores) * len(neg_scores))

# Calculate for Realistic Model
fpr, tpr = custom_roc_curve(y_true, scores_real)
auc_int = custom_auc_integration(fpr, tpr)
auc_prob = custom_auc_probabilistic(y_true, scores_real)
auc_sklearn = roc_auc_score(y_true, scores_real)

print(f"Integration Method AUC  : {auc_int:.5f}")
print(f"Probabilistic Method AUC: {auc_prob:.5f}")
print(f"Scikit-Learn AUC        : {auc_sklearn:.5f}")

ทั้งสองวิธีให้ผลลัพธ์ที่มีค่าเท่ากันทุกประการครับ!

## 💡 ความเชื่อมโยงกับคอมพิวเตอร์วิชันและ YOLO
*   **AUC ใต้เส้นโค้ง PR Curve:** คำว่า **Average Precision (AP)** คือพื้นที่ใต้เส้นโค้ง **Precision-Recall Curve** (AUC-PR) นั่นเองครับ โดย AP จะแสดงถึงคุณภาพการตรวจจับของคลาสใดคลาสหนึ่งโดยเฉพาะ
*   **mAP (mean Average Precision):** YOLO จะนำค่า AP (AUC-PR) ของทุกคลาส (เช่น ทั้ง 26 คลาสในชุดข้อมูล PTT ของคุณ) มาหาค่าเฉลี่ยเพื่อคำนวณค่า **mAP** โดยรวม (เช่น `mAP50`) สิ่งนี้ทำให้ AP/AUC เป็นเกณฑ์วัดที่สำคัญที่สุดเพียงหนึ่งเดียวในการประเมินประสิทธิภาพของตัวตรวจจับวัตถุ (object detectors) ครับ!